In [1]:
"""
draft traduccion Modelo de Liquidación de inversiones desde Access/Excel a Python
========================================



Autor: Modelos & Metodologías
Fecha: 2026-02
"""

import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta
import yaml
from pathlib import Path
import sys
import bfa_cl_utilidades as ut
import os
import pathlib
import dotenv
import datetime
import pickle
import helpers as ml_inv_helpers

fecha_proceso = 20260115

# encontremos el path del notebook actual
notebook_path = pathlib.Path().resolve()
BASE_DIR = notebook_path
# buscamos el directorio raíz del proyecto (donde está el .git)
while not (BASE_DIR / '.git').exists():
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))
# Importación de módulos internos
from config import config_rutas as cr  # Configuración de rutas del proyecto


# Carga de configuración desde archivo YAML
with open(cr.CONFIG / 'config_rutas_ext_y_archivos.yaml', 'r') as file:
    config_ext = yaml.safe_load(file)

# PROCESOS_DIARIOS_MODELOS: antiguo folder de desarrollo de modelos diarios
# vamos a definir su path, que es ~/code/PROCESOS_DIARIOS_MODELOS

procesos_diarios_modelos_path = BASE_DIR.parent / 'PROCESOS_DIARIOS_MODELOS'

data_path = procesos_diarios_modelos_path / 'data' /'external' / 'ml_inversiones'


accdb_path = data_path/"RF_Gener_Modelo_inversiones_20260115_local.accdb"
df_secuencia = pd.read_csv(data_path / "modelo_inversiones_completo_secuencia.csv",sep=";",)
df_maestro = pd.read_csv(data_path / "maestro_inversiones_funciones_completas.csv",sep=";",)


archivos_pickle = list(data_path.glob(f"tablas_linkeadas_ml_inversiones_{fecha_proceso}_*.pkl"))
if archivos_pickle:
    archivo_reciente = max(archivos_pickle, key=os.path.getctime)
    print(f"Pickle encontrado: {archivo_reciente.name}")
    
    with open(archivo_reciente, "rb") as f:
        tablas_actual = pickle.load(f)
    
    print(f"\nTablas en pickle ({len(tablas_actual)}):")
    for nombre, df in tablas_actual.items():
        print(f"  - {nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")
else:
    print("No se encontró pickle")

# Leyendo links a tablas de otros archivos que usa el access

df_links = pd.read_csv(data_path / "links_ml_inversiones.csv",sep=";",)
# Leyendo tablas del access

df_tablas = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_prod" / "RF_Gener_Modelo_Inversiones_tables.csv",sep=";",)        
# leyendo queries con dependencias del access
df_queries = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_20251121_local" / "RF_Gener_Modelo_inversiones_20251121_local_all_queries.csv",sep=";",)

queries_raw = pd.read_csv(data_path / "RF_Gener_Modelo_Inversiones_20251121_local" / "RF_Gener_Modelo_inversiones_20251121_local_queries.csv",sep=";",)


tablas = ml_inv_helpers.check_pickle_tablas_linkeadas(fecha_proceso, data_path,df_links)        
tablas.keys()


Pickle encontrado: tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl

Tablas en pickle (12):
  - FPL: 10 filas, 5 columnas
  - RF_base_Completa_Hist_Input: 414802 filas, 44 columnas
  - RF_Base_Diaria_Precios: 455659 filas, 24 columnas
  - RF_Base_Diaria_Precios_: 244010 filas, 24 columnas
  - RF_BD_Gestion_RM: 146955 filas, 23 columnas
  - RF_Cartera_RtaFija_Hist: 218556 filas, 20 columnas
  - RF_FactCLF_Banc: 1350 filas, 4 columnas
  - RF_FactCLF_Gob: 1350 filas, 4 columnas
  - RF_FactCLP_Banc: 1260 filas, 4 columnas
  - RF_FactCLP_Gob: 1260 filas, 4 columnas
  - RF_Fecha_Proceso_Carteras: 1 filas, 1 columnas
  - RF_MontosLiq: 8 filas, 5 columnas
Cargando tablas linkeadas desde pickle: C:\Users\vlandaetat\code\PROCESOS_DIARIOS_MODELOS\data\external\ml_inversiones\tablas_linkeadas_ml_inversiones_20260115_20260119_101906.pkl


dict_keys(['FPL', 'RF_base_Completa_Hist_Input', 'RF_Base_Diaria_Precios', 'RF_Base_Diaria_Precios_', 'RF_BD_Gestion_RM', 'RF_Cartera_RtaFija_Hist', 'RF_FactCLF_Banc', 'RF_FactCLF_Gob', 'RF_FactCLP_Banc', 'RF_FactCLP_Gob', 'RF_Fecha_Proceso_Carteras', 'RF_MontosLiq'])

In [2]:
from helpers import genera_cartera_inv_001,generar_cartera_instrumento, \
      generar_monto_total_instrumento, generar_cartera_pond 

In [3]:
tablas['RF_base_Completa_Hist'] = ml_inv_helpers.genera_tabla_RF_base_Completa_Hist(tablas['RF_base_Completa_Hist_Input'], fecha_proceso)

tabla_fecha_proceso = pd.DataFrame({'Fecha':[pd.to_datetime(str(fecha_proceso), format="%Y%m%d")]})
queries = {}
queries['RF_PLI_001_CarteraInv'] = genera_cartera_inv_001(
    df_base=tablas['RF_base_Completa_Hist'],
     df_fecha=tabla_fecha_proceso,
    verbose=True
)

COLUMNAS_SALIDA = [
    'Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro',
    'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 'Dias_Vcto'
]

INSTRUMENTOS = {'GobCLP':['BCP','BTP','PDB'], 'GobCLF':['BCU','BTU'],
                'DPF':['DPF'],'DPR':['DPR'], 
                'LCH':['LCH','BBC'],'BBC':['BBC'],
                               }

queries['RF_PLI_002_CarteraGobCLP'] = generar_cartera_instrumento(
    queries['RF_PLI_001_CarteraInv'],
    COLUMNAS_SALIDA,
    INSTRUMENTOS['GobCLP'],
    'GobCLP'
)

queries['RF_PLI_003_CarteraGobCLP_MonTotal'] = generar_monto_total_instrumento(
    queries['RF_PLI_002_CarteraGobCLP'],
    ['Fec_Pro', 'Cod_Pro', 'Moneda'],
    ['VP_Cap_Amort', 'VP_Int_Total'],
    'GobCLP'
)
queries['RF_PLI_004_CarteraGobCLP_Pond'] = generar_cartera_pond(
    queries['RF_PLI_002_CarteraGobCLP'],
    queries['RF_PLI_003_CarteraGobCLP_MonTotal'],
    'RF_CarteraGobCLP_Pond'
)


PASO 01b: RF_PLI_001_CarteraInv
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE 1] Nemotecnico NO empieza con 'LCH'
  Registros que cumplen: 408,778

[WHERE 2] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE 3] Cod_Sub_Pro termina en 'Disp', 'Disp_Liq' o 'MUTUOS'
  Registros que cumplen: 11,493

[WHERE 4] Clasificacion_Contable <> 'HTM'
  Registros que cumplen: 408,578

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 675

RESULTADO FINAL
  Registros entrada: 414,802
  Registros salida:  675

  Distribución Cod_Pro:
Cod_Pro
Inversion Financiera Privado    369
Inversion Financiera Publico    306

  Distribución Instrumento:
Instrumento
BBC    356
BTP    232
BTU     68
PDB      6
DPF      6
DPR      5
OTR      2
Cartera GobCLP: 238 registros después de filtrar por instrumento ['BCP', 'BTP', 'PDB']
Monto total GobCLP generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_P

In [4]:
# =============================================================================
# RECARGAR HELPERS PARA OBTENER LAS NUEVAS FUNCIONES
# =============================================================================
import importlib
import helpers
importlib.reload(helpers)

# Verificar que existe la nueva función
print("Funciones disponibles en helpers:")
funciones_relevantes = [f for f in dir(helpers) if 'flujo' in f.lower() or 'configuracion' in f.lower()]
for func in funciones_relevantes:
    print(f" - {func}")

# Verificar configuración de instrumentos
print("\n" + "="*50)
print("CONFIGURACIÓN DE INSTRUMENTOS DISPONIBLES:")
print("="*50)
for inst, config in helpers.CONFIGURACION_INSTRUMENTOS.items():
    print(f"\n{inst}:")
    print(f"  - Códigos disp: {config['codigos_disp']}")
    print(f"  - Códigos pacto: {config['codigos_pacto']}")
    print(f"  - Tabla factores: {config['tabla_factores']}")

Funciones disponibles en helpers:
 - CONFIGURACION_INSTRUMENTOS
 - calcular_flujo_liquidacion
 - generar_flujo_liquidacion_instrumento

CONFIGURACIÓN DE INSTRUMENTOS DISPONIBLES:

GobCLP:
  - Códigos disp: ['BCP', 'BTP', 'PDB']
  - Códigos pacto: ['BCP', 'BTP', 'PDB']
  - Tabla factores: RF_FactCLP_Gob

GobCLF:
  - Códigos disp: ['BCU', 'BTU']
  - Códigos pacto: ['BCU', 'BTU', 'CER']
  - Tabla factores: RF_FactCLF_Gob

DPF:
  - Códigos disp: ['DPF']
  - Códigos pacto: ['DPF', 'FFM']
  - Tabla factores: RF_FactCLP_Banc

DPR:
  - Códigos disp: ['DPR']
  - Códigos pacto: ['DPR']
  - Tabla factores: RF_FactCLF_Banc

BBC:
  - Códigos disp: ['BBC']
  - Códigos pacto: ['BBC']
  - Tabla factores: RF_FactCLP_Banc

LCH:
  - Códigos disp: ['LCH', 'BBC']
  - Códigos pacto: ['LCH', 'BBC']
  - Tabla factores: RF_FactCLF_Banc


In [5]:
# TODO: agregar DPX (en USD) con los parámetros de DPR, pero con montos a liquidar en dólares. 


# Fase 0: Verificación de Tablas Base
Verificamos estructura de las tablas linkeadas necesarias para las queries de haircut y liquidación.

In [6]:
# =============================================================================
# FASE 0b: Limpieza de Tablas Base
# =============================================================================

print("="*70)
print("LIMPIEZA DE TABLAS BASE")
print("="*70)

# 1. FPL - Limpiar columnas basura y filas NaN
print("\n[1] Limpiando FPL...")
tablas['FPL'] = tablas['FPL'][['Instrumento', 'Haircut']].dropna()
print(f"    FPL limpio: {len(tablas['FPL'])} registros")
display(tablas['FPL'])

# 2. RF_MontosLiq - Re-estructurar (headers en fila 0, col 0 vacía)
print("\n[2] Re-estructurando RF_MontosLiq...")
df_montos_raw = tablas['RF_MontosLiq'].copy()
# Tomar desde la segunda columna (ignorar col 0 vacía)
df_montos_clean = df_montos_raw.iloc[1:, 1:].copy()  # Desde fila 1, desde col 1
# Usar la primera fila como headers
df_montos_clean.columns = df_montos_raw.iloc[0, 1:].values
# Resetear índice
df_montos_clean = df_montos_clean.reset_index(drop=True)
# Convertir columnas numéricas
for col in ['Monto Mercado', '% participacion', 'Monto a Liquidar']:
    df_montos_clean[col] = pd.to_numeric(df_montos_clean[col], errors='coerce')
# Guardar tabla limpia
tablas['RF_MontosLiq'] = df_montos_clean
print(f"    RF_MontosLiq limpio: {len(tablas['RF_MontosLiq'])} registros")
display(tablas['RF_MontosLiq'])

print("\n" + "="*70)
print("✓ FASE 0 COMPLETADA - Tablas listas para usar")
print("="*70)

LIMPIEZA DE TABLAS BASE

[1] Limpiando FPL...
    FPL limpio: 7 registros


,Instrumento,Haircut
0,Gobierno CLP,0.002862
1,Gobierno CLF,0.006937
2,DPF,0.004804
3,DPR,0.001980
4,Corporativo CLP,0.012151
5,Corporativo CLF,0.007283
6,LCHR,0.010844



[2] Re-estructurando RF_MontosLiq...
    RF_MontosLiq limpio: 7 registros


,Instrumento,Monto Mercado,% participacion,Monto a Liquidar
0,Gobierno CLP,9.762388e+11,0.113718,1.110155e+11
1,Gobierno CLF,3.114182e+06,0.236630,7.369081e+05
2,DPF,3.030708e+11,0.090793,2.751656e+10
3,DPR,4.086030e+05,0.264012,1.078761e+05
4,Corporativo CLP,1.285508e+10,0.131426,1.689491e+09
5,Corporativo CLF,3.820034e+06,0.136548,5.216180e+05
6,LCHR,1.655843e+04,0.136548,0.000000e+00



✓ FASE 0 COMPLETADA - Tablas listas para usar


# Fase 1: Rama PACTOS
Generamos las queries de la rama de pactos:
- `RF_PLI_001d_CarteraInv_Pcto` 
- `RF_PLI_002_CarteraGobCLP_Pacto`
- `RF_PLI_003b_GobCLP_MontoPlazo_Pacto`

In [7]:
# =============================================================================
# FASE 1: RAMA PACTOS
# =============================================================================
# Recargamos helpers para usar las nuevas funciones
import importlib
import helpers as ml_inv_helpers
importlib.reload(ml_inv_helpers)

from helpers import genera_cartera_inv_pacto, generar_monto_plazo_pacto

# -----------------------------------------------------------------------------
# 1.1 RF_PLI_001d_CarteraInv_Pcto - Cartera de inversiones con pactos
# -----------------------------------------------------------------------------
queries['RF_PLI_001d_CarteraInv_Pcto'] = genera_cartera_inv_pacto(
    df_base=tablas['RF_base_Completa_Hist'],
    df_fecha=tabla_fecha_proceso,
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_001d_CarteraInv_Pcto'].head())


FASE 1.1: RF_PLI_001d_CarteraInv_Pcto (Cartera Pactos)
Registros entrada: 414,802

[JOIN] Filtro fecha proceso = 2026-01-15
  Registros que cumplen: 51,748

[WHERE 1] Cod_Pro es inversión financiera
  Registros que cumplen: 11,718

[WHERE 2] Cod_Sub_Pro termina en 'Pcto' o 'Pcto_Liq'
  Registros que cumplen: 0

[WHERE FINAL] Todos los filtros combinados (AND)
  Registros que cumplen: 0

RESULTADO: 0 registros generados
Columnas: ['Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro', 'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 'Dias_Vcto', 'Dias_Pacto']

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Dias_Pacto


In [8]:
# -----------------------------------------------------------------------------
# 1.2 RF_PLI_002_CarteraGobCLP_Pacto - Filtrar por instrumento GobCLP
# -----------------------------------------------------------------------------
# Reutilizamos generar_cartera_instrumento pero con columnas que incluyen Dias_Pacto

COLUMNAS_SALIDA_PACTO = [
    'Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro',
    'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total', 
    'Dias_Vcto', 'Dias_Pacto'
]

queries['RF_PLI_002_CarteraGobCLP_Pacto'] = generar_cartera_instrumento(
    queries['RF_PLI_001d_CarteraInv_Pcto'],
    COLUMNAS_SALIDA_PACTO,
    INSTRUMENTOS['GobCLP'],  # ['BCP', 'BTP', 'PDB']
    'GobCLP_Pacto'
)

print("\nPrimeras filas:")
display(queries['RF_PLI_002_CarteraGobCLP_Pacto'].head())


# -----------------------------------------------------------------------------
# 1.3 RF_PLI_003b_GobCLP_MontoPlazo_Pacto - Monto por plazo de pacto
# -----------------------------------------------------------------------------
queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'] = generar_monto_plazo_pacto(
    queries['RF_PLI_002_CarteraGobCLP_Pacto'],
    verbose=True
)

print("\nResultado completo:")
display(queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'])

print("\n" + "="*70)
print("✓ FASE 1 COMPLETADA - Rama PACTOS lista")
print("="*70)
print(f"\nQueries generadas:")
print(f"  - RF_PLI_001d_CarteraInv_Pcto: {len(queries['RF_PLI_001d_CarteraInv_Pcto']):,} registros")
print(f"  - RF_PLI_002_CarteraGobCLP_Pacto: {len(queries['RF_PLI_002_CarteraGobCLP_Pacto']):,} registros")
print(f"  - RF_PLI_003b_GobCLP_MontoPlazo_Pacto: {len(queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto']):,} registros")

Cartera GobCLP_Pacto: 0 registros después de filtrar por instrumento ['BCP', 'BTP', 'PDB']

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Dias_Pacto



Resultado completo:


,Dias_Pacto,Monto



✓ FASE 1 COMPLETADA - Rama PACTOS lista

Queries generadas:
  - RF_PLI_001d_CarteraInv_Pcto: 0 registros
  - RF_PLI_002_CarteraGobCLP_Pacto: 0 registros
  - RF_PLI_003b_GobCLP_MontoPlazo_Pacto: 0 registros


# Fase 2: Rama HAIRCUT
Generamos las queries de haircut:
- `RF_PLI_005_CarteraHC` - Cartera con factores de haircut
- `RF_PLI_006_Haircut_Dia` - Haircut agregado por día
- `RF_PLI_006b_Haircut_Dia` - Haircut con día de semana

In [9]:
# =============================================================================
# FASE 2: RAMA HAIRCUT
# =============================================================================
# Recargamos helpers para usar las nuevas funciones
importlib.reload(ml_inv_helpers)

from helpers import (
    generar_cartera_haircut, 
    generar_haircut_dia, 
    agregar_dia_semana,
    combinar_haircut_con_pactos,
    filtrar_monto_liquidar
)

# -----------------------------------------------------------------------------
# 2.1 RF_PLI_005_CarteraHC - Cartera con factores de haircut
# -----------------------------------------------------------------------------
queries['RF_PLI_005_CarteraHC'] = generar_cartera_haircut(
    df_cartera_pond=queries['RF_PLI_004_CarteraGobCLP_Pond'],
    df_factores=tablas['RF_FactCLP_Gob'],
    df_fpl=tablas['FPL'],
    filtro_instrumento='Gobierno CLP',
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_005_CarteraHC'].head())


FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 238
Registros factores: 1,260

[FPL] Haircut para '['Gobierno CLP']': 0.002862

[JOIN] Registros después de JOIN con factores: 21,420

[CALC] FactorPond calculado usando MAX(Factor, 0.002862)
  Factor min: 0.000933
  Factor max: 0.059951

RESULTADO: 21,420 registros generados

Primeras filas:


,Fec_Pro,Cod_Emp,Moneda,Cod_Pro,Cod_Sub_Pro,Nemotecnico,Instrumento,VP_Cap_Amort,VP_Int_Total,Dias_Vcto,Ponderador,Dia,Factor,FactorPond
0,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,1,0.000933,3.933746e-07
1,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,2,0.001210,3.933746e-07
2,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,3,0.001487,3.933746e-07
3,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,4,0.001764,3.933746e-07
4,2026-01-15,1.0,CLP,Inversion Financiera Publico,Inversion Financiera Publico_TGR_Disp,BTP0581029,BTP,0.0,8.612043e+07,76.0,0.000137,5,0.002041,3.933746e-07


In [10]:
# -----------------------------------------------------------------------------
# 2.2 RF_PLI_006_Haircut_Dia - Haircut agregado por día
# -----------------------------------------------------------------------------
queries['RF_PLI_006_Haircut_Dia'] = generar_haircut_dia(
    queries['RF_PLI_005_CarteraHC'],
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_006_Haircut_Dia'].head(10))


--------------------------------------------------
FASE 2.2: RF_PLI_0XX_Haircut_Dia
--------------------------------------------------
Registros entrada: 21,420
Registros salida: 90
Días: 1 a 90

Primeras filas:


,Dia,Haircut
0,1,0.004047
1,2,0.004205
2,3,0.004362
3,4,0.004520
4,5,0.004679
5,6,0.004838
6,7,0.004997
7,8,0.005254
8,9,0.005510
9,10,0.005766


In [11]:
# -----------------------------------------------------------------------------
# 2.3 RF_PLI_006b_Haircut_Dia - Haircut con día de semana
# -----------------------------------------------------------------------------
queries['RF_PLI_006b_Haircut_Dia'] = agregar_dia_semana(
    queries['RF_PLI_006_Haircut_Dia'],
    fecha_proceso=fecha_proceso,
    verbose=True
)

print("\nPrimeras filas (DiaSem: 1=Lun, ..., 7=Dom):")
display(queries['RF_PLI_006b_Haircut_Dia'].head(10))

print("\n" + "="*70)
print("✓ FASE 2 COMPLETADA - Rama HAIRCUT lista")
print("="*70)


--------------------------------------------------
FASE 2.3: RF_PLI_0XXb_Haircut_Dia (con día semana)
--------------------------------------------------
Registros entrada: 90
Fecha proceso: 2026-01-15 (Jue)
Registros salida: 90

Primeras filas (DiaSem: 1=Lun, ..., 7=Dom):


,Dia,DiaSem,Haircut
0,1,5,0.004047
1,2,6,0.004205
2,3,7,0.004362
3,4,1,0.004520
4,5,2,0.004679
5,6,3,0.004838
6,7,4,0.004997
7,8,5,0.005254
8,9,6,0.005510
9,10,7,0.005766



✓ FASE 2 COMPLETADA - Rama HAIRCUT lista


# Fase 3: Combinación Haircut + Pactos
Combinamos la rama de haircut con los montos de pactos para generar:
- `RF_PLI_006c_Haircut_Dia_Pcto`

In [12]:
# =============================================================================
# FASE 3: COMBINACIÓN HAIRCUT + PACTOS
# =============================================================================
queries['RF_PLI_006c_Haircut_Dia_Pcto'] = combinar_haircut_con_pactos(
    df_haircut_dia_sem=queries['RF_PLI_006b_Haircut_Dia'],
    df_monto_plazo_pacto=queries['RF_PLI_003b_GobCLP_MontoPlazo_Pacto'],
    verbose=True
)

print("\nPrimeras filas:")
display(queries['RF_PLI_006c_Haircut_Dia_Pcto'].head(10))


FASE 3: RF_PLI_0XXc_Haircut_Dia_Pcto (Haircut + Pactos)
Registros haircut: 90
Registros pactos: 0

Registros salida: 90
Días con pactos: 0

Primeras filas:


,Dia,DiaSem,Haircut,Monto_Pacto
0,1,5,0.004047,0.0
1,2,6,0.004205,0.0
2,3,7,0.004362,0.0
3,4,1,0.004520,0.0
4,5,2,0.004679,0.0
5,6,3,0.004838,0.0
6,7,4,0.004997,0.0
7,8,5,0.005254,0.0
8,9,6,0.005510,0.0
9,10,7,0.005766,0.0


# Fase 4: Monto a Liquidar
Obtenemos el monto a liquidar para el instrumento:
- `RF_PLI_007_MontoLiquidar`

In [13]:
# =============================================================================
# FASE 4: MONTO A LIQUIDAR
# =============================================================================
queries['RF_PLI_007_MontoLiquidar'] = filtrar_monto_liquidar(
    df_montos_liq=tablas['RF_MontosLiq'],
    instrumento='Gobierno CLP',
    verbose=True
)

display(queries['RF_PLI_007_MontoLiquidar'])


--------------------------------------------------
FASE 4: RF_PLI_0XX_MontoLiquidar (Gobierno CLP)
--------------------------------------------------
Monto a liquidar: 111,015,499,404.70


,Instrumento,Monto Mercado,% participacion,Monto a Liquidar
0,Gobierno CLP,9.762388e+11,0.113718,1.110155e+11


# Fase 5: Integración - Ejecutar monto_liq_gob_clp
Ejecutamos la función principal con los 3 inputs generados:
1. `RF_PLI_003_CarteraGobCLP_MonTotal` (monto total)
2. `RF_PLI_006c_Haircut_Dia_Pcto` (haircut + pactos)
3. `RF_PLI_007_MontoLiquidar` (monto a liquidar)

In [14]:
# =============================================================================
# FASE 5: INTEGRACIÓN - EJECUTAR FUNCIÓN PRINCIPAL
# =============================================================================
# Recargar helpers con la corrección
importlib.reload(ml_inv_helpers)
from helpers import monto_liq_gob_clp

print("="*70)
print("FASE 5: INTEGRACIÓN - Ejecutando monto_liq_gob_clp()")
print("="*70)

print("\nInputs:")
print(f"  1. RF_PLI_003_CarteraGobCLP_MonTotal: {len(queries['RF_PLI_003_CarteraGobCLP_MonTotal'])} registros")
print(f"     VP_Flujo = {queries['RF_PLI_003_CarteraGobCLP_MonTotal']['VP_Flujo'].sum():,.2f}")
print(f"  2. RF_PLI_006c_Haircut_Dia_Pcto: {len(queries['RF_PLI_006c_Haircut_Dia_Pcto'])} registros")
print(f"  3. RF_PLI_007_MontoLiquidar: {len(queries['RF_PLI_007_MontoLiquidar'])} registros")
print(f"     Monto a Liquidar (diario): {queries['RF_PLI_007_MontoLiquidar']['Monto a Liquidar'].iloc[0]:,.2f}")

# Ejecutar función principal
flujo_gob_clp = monto_liq_gob_clp(
    df_cartera_mon_total=queries['RF_PLI_003_CarteraGobCLP_MonTotal'],
    df_haircut_dia_pcto=queries['RF_PLI_006c_Haircut_Dia_Pcto'],
    df_monto_liquidar=queries['RF_PLI_007_MontoLiquidar']
)

print("\n" + "="*70)
print("RESULTADO: Flujo_GobCLP")
print("="*70)
print(f"Registros generados: {len(flujo_gob_clp)}")
display(flujo_gob_clp.head(15))
display(flujo_gob_clp.tail(10))

FASE 5: INTEGRACIÓN - Ejecutando monto_liq_gob_clp()

Inputs:
  1. RF_PLI_003_CarteraGobCLP_MonTotal: 1 registros
     VP_Flujo = 626,643,809,581.59
  2. RF_PLI_006c_Haircut_Dia_Pcto: 90 registros
  3. RF_PLI_007_MontoLiquidar: 1 registros
     Monto a Liquidar (diario): 111,015,499,404.70

RESULTADO: Flujo_GobCLP
Registros generados: 91


,Dia,DiaSem,Haircut,Monto_Liquidar
0,0.0,NaN,0.000000,6.266438e+11
1,1.0,5.0,0.004047,1.110155e+11
2,2.0,6.0,0.004205,0.000000e+00
3,3.0,7.0,0.004362,0.000000e+00
4,4.0,1.0,0.004520,1.110155e+11
5,5.0,2.0,0.004679,1.110155e+11
6,6.0,3.0,0.004838,1.110155e+11
7,7.0,4.0,0.004997,1.110155e+11
8,8.0,5.0,0.005254,6.862433e+10
9,9.0,6.0,0.005510,0.000000e+00


,Dia,DiaSem,Haircut,Monto_Liquidar
81,81.0,1.0,0.015046,0.0
82,82.0,2.0,0.015145,0.0
83,83.0,3.0,0.015244,0.0
84,84.0,4.0,0.015342,0.0
85,85.0,5.0,0.015441,0.0
86,86.0,6.0,0.015539,0.0
87,87.0,7.0,0.015638,0.0
88,88.0,1.0,0.015736,0.0
89,89.0,2.0,0.015835,0.0
90,90.0,3.0,0.015933,0.0


In [15]:
# =============================================================================
# EJECUTAR PIPELINE GOBCLF (GOBIERNO CLF - BONOS UF)
# =============================================================================

# Preparar diccionario de tablas necesarias
tablas_gobclf = {
    'RF_FactCLF_Gob': tablas['RF_FactCLF_Gob'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_gob_clf, queries_gobclf = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_gobclf,
    tipo_instrumento='GobCLF',
    fecha_proceso=fecha_proceso,
    verbose=True
)

print("\n" + "="*70)
print("RESUMEN FLUJO GobCLF")
print("="*70)
display(flujo_gob_clf.head(15))


PIPELINE DE LIQUIDACIÓN: Gobierno CLF
Tipo: GobCLF
Códigos disponible: ['BCU', 'BTU']
Códigos pacto: ['BCU', 'BTU', 'CER']
Tabla factores: RF_FactCLF_Gob
Instrumento FPL: Gobierno CLF
Cartera GobCLF: 68 registros después de filtrar por instrumento ['BCU', 'BTU']

Cartera filtrada para GobCLF: 68 registros
Monto total GobCLF generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para GobCLF: 1 registros
Cartera ponderada RF_CarteraGobCLF_Pond: 68 registros generados.

Cartera ponderada para GobCLF: 68 registros
Cartera GobCLF_Pacto: 0 registros después de filtrar por instrumento ['BCU', 'BTU', 'CER']

Cartera pactos filtrada para GobCLF: 0 registros

Monto por plazo de pacto para GobCLF: 0 registros

Usando tabla de factores: RF_FactCLF_Gob (1,350 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 68
Registros factores: 1,350

[FPL] Haircut para '['Gobierno CLF']': 0.006937

[JOIN] Registros después 

,Dia,DiaSem,Haircut,Monto_Liquidar
0,0.0,NaN,0.000000,1.094037e+06
1,1.0,5.0,0.006937,7.369081e+05
2,2.0,6.0,0.008281,0.000000e+00
3,3.0,7.0,0.010069,0.000000e+00
4,4.0,1.0,0.011858,3.477801e+05
5,5.0,2.0,0.013647,0.000000e+00
6,6.0,3.0,0.015436,0.000000e+00
7,7.0,4.0,0.017224,0.000000e+00
8,8.0,5.0,0.018647,0.000000e+00
9,9.0,6.0,0.020069,0.000000e+00


In [16]:
# =============================================================================
# EJECUTAR PIPELINE DPF (DEPOSITOS A PLAZO FIJO)
# =============================================================================
# =============================================================================
import importlib
import helpers
importlib.reload(helpers)

# Preparar diccionario de tablas necesarias
tablas_dpf = {
    'RF_FactCLP_Banc': tablas['RF_FactCLP_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}
# Ejecutar pipeline completo
flujo_dpf, queries_dpf = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_dpf,
    tipo_instrumento='DPF',
    fecha_proceso=fecha_proceso,
    verbose=True
)


PIPELINE DE LIQUIDACIÓN: Depósito a Plazo Fijo
Tipo: DPF
Códigos disponible: ['DPF']
Códigos pacto: ['DPF', 'FFM']
Tabla factores: RF_FactCLP_Banc
Instrumento FPL: DPF
Cartera DPF: 6 registros después de filtrar por instrumento ['DPF']

Cartera filtrada para DPF: 6 registros
Monto total DPF generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para DPF: 1 registros
Cartera ponderada RF_CarteraDPF_Pond: 6 registros generados.

Cartera ponderada para DPF: 6 registros
Cartera DPF_Pacto: 0 registros después de filtrar por instrumento ['DPF', 'FFM']

Cartera pactos filtrada para DPF: 0 registros

Monto por plazo de pacto para DPF: 0 registros

Usando tabla de factores: RF_FactCLP_Banc (1,260 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 6
Registros factores: 1,260

[FPL] Haircut para '['DPF']': 0.004804

[JOIN] Registros después de JOIN con factores: 540

[CALC] FactorPond calculado usando MAX(Fact

In [17]:
tablas_dpr = {
    'RF_FactCLF_Banc': tablas['RF_FactCLF_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_dpr, queries_dpr = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_dpr,
    tipo_instrumento='DPR',
    fecha_proceso=fecha_proceso,
    verbose=True
)

tablas_bbc = {
    'RF_FactCLP_Banc': tablas['RF_FactCLP_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}
# Ejecutar pipeline completo
flujo_bbc, queries_bbc = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_bbc,
    tipo_instrumento='BBC',
    fecha_proceso=fecha_proceso,
    verbose=True
)

tablas_lch = {
    'RF_FactCLF_Banc': tablas['RF_FactCLF_Banc'],
    'FPL': tablas['FPL'],
    'RF_MontosLiq': tablas['RF_MontosLiq']
}

# Ejecutar pipeline completo
flujo_lch, queries_lch = helpers.generar_flujo_liquidacion_instrumento(
    df_cartera_inv=queries['RF_PLI_001_CarteraInv'],
    df_cartera_inv_pacto=queries['RF_PLI_001d_CarteraInv_Pcto'],
    tablas=tablas_lch,
    tipo_instrumento='LCH',
    fecha_proceso=fecha_proceso,
    verbose=True
)
# =============================================================================


PIPELINE DE LIQUIDACIÓN: Depósito a Plazo Reajustable
Tipo: DPR
Códigos disponible: ['DPR']
Códigos pacto: ['DPR']
Tabla factores: RF_FactCLF_Banc
Instrumento FPL: DPR
Cartera DPR: 5 registros después de filtrar por instrumento ['DPR']

Cartera filtrada para DPR: 5 registros
Monto total DPR generado: 1 registros después de agrupar por ['Fec_Pro', 'Cod_Pro', 'Moneda']

Monto total calculado para DPR: 1 registros
Cartera ponderada RF_CarteraDPR_Pond: 5 registros generados.

Cartera ponderada para DPR: 5 registros
Cartera DPR_Pacto: 0 registros después de filtrar por instrumento ['DPR']

Cartera pactos filtrada para DPR: 0 registros

Monto por plazo de pacto para DPR: 0 registros

Usando tabla de factores: RF_FactCLF_Banc (1,350 registros)

FASE 2.1: RF_PLI_0XX_CarteraHC (Cartera con Haircut)
Registros cartera entrada: 5
Registros factores: 1,350

[FPL] Haircut para '['DPR']': 0.001980

[JOIN] Registros después de JOIN con factores: 450

[CALC] FactorPond calculado usando MAX(Factor, 0.0

In [18]:
# TODO: agregar DPX (en USD) con los parámetros de DPR, pero con montos a liquidar en dólares. 


In [19]:
# =============================================================================
# COMPARACION DE FLUJOS OBTENIDOS VS FLUJOS EN ACCESS
# =============================================================================

flujos = {
    'GobCLP': flujo_gob_clp,
    'GobCLF': flujo_gob_clf,
    'DPF': flujo_dpf,
    'DPR': flujo_dpr,
    'BBC': flujo_bbc,
    'LCH': flujo_lch
    
    }

# leer tablas access local, que están en .pkl en data_path
with open(data_path / f"tablas_access_prod_20260115_20260119_102556.pkl", "rb") as f:
    tablas_access = pickle.load(f)
flujos_access = {
    'GobCLP': tablas_access['Flujo_GobCLP'],
    'GobCLF': tablas_access['Flujo_GobCLF'],
    'DPF': tablas_access['Flujo_DPF'],
    'DPR': tablas_access['Flujo_DPR'],
    'BBC': tablas_access['Flujo_BBC'],
    'LCH': tablas_access['Flujo_LCH']
    }



In [20]:


# comparar cada flujo generado con el de access

for inst in flujos.keys():
    print("\n" + "="*70)
    print(f"Comparando flujo {inst}")
    print("="*70)
    df_gen = flujos[inst]
    df_acc = flujos_access[inst]
    
    # comparar diferencias por día en Monto_Liquidar
    df_comp = pd.merge(
        df_gen.groupby('Dia').sum().reset_index(),
        df_acc.groupby('Dia').sum().reset_index(),
        on='Dia',
        suffixes=('_gen', '_acc')
    )
    # diferencias en cada columna:
    for col in ['Monto_Liquidar', 'Haircut']:
        df_comp[f'Diff_{col}'] = df_comp[f'{col}_gen'] - df_comp[f'{col}_acc']


    # log de resumen de diferencias
    total_diff = df_comp[[f'Diff_{col}' for col in ['Monto_Liquidar', 'Haircut',]]].sum()
    print(f"Resumen diferencias totales:")
    for col in ['Monto_Liquidar', 'Haircut']:
        print(f"  - {col}: {total_diff[f'Diff_{col}']:}")
    print("="*70)
    print("Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):")
    diffs_significativos = df_comp[
        (df_comp['Diff_Monto_Liquidar'].abs() > 1) |
        (df_comp['Diff_Haircut'].abs() > 0.0001)
    ]
    if diffs_significativos.empty:
        print("  - No se encontraron diferencias significativas.")
    else:
        display(diffs_significativos)


Comparando flujo GobCLP
Resumen diferencias totales:
  - Monto_Liquidar: -3.0517578125e-05
  - Haircut: -3.8163916471489756e-17
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo GobCLF
Resumen diferencias totales:
  - Monto_Liquidar: -4.656612873077393e-10
  - Haircut: 4.891920202254596e-16
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo DPF
Resumen diferencias totales:
  - Monto_Liquidar: 0.0
  - Haircut: 1.0408340855860843e-17
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se encontraron diferencias significativas.

Comparando flujo DPR
Resumen diferencias totales:
  - Monto_Liquidar: 0.0
  - Haircut: 8.239936510889834e-18
Días con diferencias significativas (> 1 peso para Monto_Liquidar, > 0.01 para Haircut):
  - No se en

De la secuencia inicial de 27 pasos, estamos ok hasta el paso 19. El paso 20 es la query RF_PLI_045_Gener_Precios_Dia, que es una consulta sencilla a algunas columnas de la tabla de precios.
La consulta es:
```sql
SELECT RF_Base_Diaria_Precios.Fecha, RF_Base_Diaria_Precios.NEMOTECNICO, RF_Base_Diaria_Precios.Instrumento, RF_Base_Diaria_Precios.Precio_Mid INTO Precios_Dia
FROM RF_Fecha_Proceso_Carteras INNER JOIN RF_Base_Diaria_Precios ON RF_Fecha_Proceso_Carteras.Fecha = RF_Base_Diaria_Precios.Fecha
WHERE (((RF_Base_Diaria_Precios.Instrumento)="TCRC"));
```

In [21]:
#print(queries_raw.loc[queries_raw['nombre']=='RF_PLI_045_Gener_Precios_Dia','sql'].values[0])

# RF_Base_Diaria_Precios es una tabla externa, que se llama de la misma forma,
#  en el archivo access '\\\\vmdvorak\\Riesgo Financiero2\\RF_PROCESOS\\RF_Proceso_Tasas\\RF_Base_PT.accdb'
# y que está en el diccionario tablas
columnas_RF_PLI_045_Gener_Precios_Dia = ['Fecha','NEMOTECNICO','Instrumento','Precio_Mid']
filtro_RF_PLI_045_Gener_Precios_Dia = 'Instrumento=="TCRC"'
queries['RF_PLI_045_Gener_Precios_Dia'] = tablas['RF_Base_Diaria_Precios'][columnas_RF_PLI_045_Gener_Precios_Dia].query(
    'Fecha==@fecha_proceso').query(filtro_RF_PLI_045_Gener_Precios_Dia)
queries['RF_PLI_045_Gener_Precios_Dia']


,Fecha,NEMOTECNICO,Instrumento,Precio_Mid
454136,2026-01-15,AUD,TCRC,591.990016
454137,2026-01-15,COP,TCRC,0.239949
454138,2026-01-15,BRL,TCRC,164.697105
454139,2026-01-15,CAD,TCRC,635.463443
454140,2026-01-15,CHF,TCRC,1099.813177
454141,2026-01-15,CLF,TCRC,39747.120000
454142,2026-01-15,CLP,TCRC,1.000000
454143,2026-01-15,CNY,TCRC,126.767923
454144,2026-01-15,ARS,TCRC,0.611261
454145,2026-01-15,DKK,TCRC,137.205364


El paso 21 es RF_PLI_044e_Modelo_Inversiones_Tabla_Final. Sin embargo, esta query depende de otra query, y así de manera anidada, tenemos un total de 12 queries que componen el paso 21. Leyendo el listado de queries de abajo hacia arriba, podemos ver las dependencias.

## Listado de Queries

🔹 **RF_PLI_001b_CarteraInv_Gtia** (Select)

🔹 **RF_PLI_001c_CarteraInv_Gtia** (Select)
   - Depende de: RF_PLI_001b_CarteraInv_Gtia

🔹 **RF_PLI_008b_CarteraGobCLP_Final** (Select)

🔹 **RF_PLI_015b_CarteraGobCLF_Final** (Select)

🔹 **RF_PLI_022b_CarteraDPF_Final** (Select)

🔹 **RF_PLI_029b_CarteraDPR_Final** (Select)

🔹 **RF_PLI_036b_CarteraLCH_Final** (Select)

🔹 **RF_PLI_043b_CarteraBBC_Final** (Select)

🔹 **RF_PLI_044_Modelo_Inversiones_Final** (Type128)
   - Depende de: RF_PLI_001c_CarteraInv_Gtia, RF_PLI_008b_CarteraGobCLP_Final, RF_PLI_015b_CarteraGobCLF_Final, RF_PLI_022b_CarteraDPF_Final, RF_PLI_029b_CarteraDPR_Final, RF_PLI_036b_CarteraLCH_Final, RF_PLI_043b_CarteraBBC_Final

🔹 **RF_PLI_044c_Modelo_Inversiones_Pacto_FB** (Select)

🔹 **RF_PLI_044d_Modelo_Inversiones_Full** (Type128)
   - Depende de: RF_PLI_044_Modelo_Inversiones_Final, RF_PLI_044c_Modelo_Inversiones_Pacto_FB

🎯 **RF_PLI_044e_Modelo_Inversiones_Tabla_Final** (DDL)
   - Depende de: RF_PLI_044d_Modelo_Inversiones_Full

In [24]:
df_secuencia.tail(6)
queries_raw

,nombre,tipo,sql,hash_sha1
0,borrar,Select,SELECT *\r\nFROM RF_Fecha_Proceso_Carteras;\r\n,100922308adb9f3de19e4309be22c4cd82a96848
1,Consulta_Montos_para_LIQ_VP_Flujo_Cartera,Select,SELECT *\r\nFROM Montos_para_LIQ_VP_Flujo_Cart...,360f19d60d67a387878a54e48328478e1f7d1ebc
2,Montos_para_LIQ_VP_Flujo_Cartera,Type128,SELECT RF_PLI_003_CarteraGobCLP_MonTotal.Fec_P...,c3244fdf842665e77a2788f3917e809afaa82789
3,PRUEBA_BORRAR,Select,SELECT *\r\nFROM RF_base_Completa_Hist_Input;\r\n,53821ffa4982f01e5c2a8ffc1c11012feddf9a1b
4,RF_PLI_000_Gener_CarteraInv,DDL,SELECT RF_base_Completa_Hist_Input.* INTO RF_b...,fc0b569bb960193a25551fe9a9fd887db9aa0cf4
...,...,...,...,...
113,RF_PLI_048b_Tabla_Desarrollo_Interna_Add_HTM,Type64,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,8b1744445bb7d096d042442ffaa3099baca01e34
114,RF_PLI_048c_Tabla_Desarrollo_Interna_Add_RT,Type64,INSERT INTO RF_Tabla_Desarrollo_Interna ( Fec_...,6dbac1bc1bafd9139736621409be751d538925bf
115,RF_PLI_049_Tabla_Desarrollo_Modelo_Inversiones,Type128,SELECT \r\nRF_PLI_046_Modelo_Inversiones_Final...,2bb37e4cf2479d921bb3735e63a64766325acf9a
116,RF_PLI_050_Tabla_Desarrollo_Modelo_Inversiones...,DDL,SELECT RF_PLI_049_Tabla_Desarrollo_Modelo_Inve...,edd8d07a6a4a21a3031106524fa1771bed63319d


In [ ]:
queries_finales = df_secuencia[df_secuencia['paso']>20]['nombre_accion'].to_list()

In [25]:
# =============================================================================
# ANÁLISIS DE LAS 12 QUERIES DEL PASO 21
# =============================================================================

# Lista de las 12 queries que componen el paso 21
queries_paso_21 = [
    'RF_PLI_001b_CarteraInv_Gtia',
    'RF_PLI_001c_CarteraInv_Gtia',
    'RF_PLI_008b_CarteraGobCLP_Final',
    'RF_PLI_015b_CarteraGobCLF_Final',
    'RF_PLI_022b_CarteraDPF_Final',
    'RF_PLI_029b_CarteraDPR_Final',
    'RF_PLI_036b_CarteraLCH_Final',
    'RF_PLI_043b_CarteraBBC_Final',
    'RF_PLI_044_Modelo_Inversiones_Final',
    'RF_PLI_044c_Modelo_Inversiones_Pacto_FB',
    'RF_PLI_044d_Modelo_Inversiones_Full',
    'RF_PLI_044e_Modelo_Inversiones_Tabla_Final'
]

# Extraer el SQL de cada query
print("="*80)
print("ANÁLISIS DETALLADO DE LAS 12 QUERIES DEL PASO 21")
print("="*80)

analisis_queries = {}
for query_name in queries_paso_21:
    query_row = queries_raw[queries_raw['nombre'] == query_name]
    if not query_row.empty:
        sql = query_row['sql'].values[0]
        tipo = query_row['tipo'].values[0] if 'tipo' in query_row.columns else 'N/A'
        analisis_queries[query_name] = {
            'sql': sql,
            'tipo': tipo
        }
        print(f"\n{'='*80}")
        print(f"📋 {query_name}")
        print(f"   Tipo: {tipo}")
        print(f"{'='*80}")
        print(sql[:2000] if len(sql) > 2000 else sql)
        print(f"\n   [Longitud SQL: {len(sql)} caracteres]")
    else:
        print(f"\n⚠️ Query no encontrada: {query_name}")

ANÁLISIS DETALLADO DE LAS 12 QUERIES DEL PASO 21

📋 RF_PLI_001b_CarteraInv_Gtia
   Tipo: Select
SELECT RF_base_Completa_Hist_Input.Fec_Pro, RF_base_Completa_Hist_Input.Cod_Emp, RF_base_Completa_Hist_Input.Moneda, RF_base_Completa_Hist_Input.Cod_Pro, RF_base_Completa_Hist_Input.Cod_Sub_Pro, RF_base_Completa_Hist_Input.Nemotecnico, Left(RF_base_Completa_Hist_Input.Nemotecnico,3) AS Instrumento, RF_base_Completa_Hist_Input.Cap_Amort, RF_base_Completa_Hist_Input.Int_Total_Cont, RF_base_Completa_Hist_Input.Dias_Vcto, RF_base_Completa_Hist_Input.VP_Cap_Amort, RF_base_Completa_Hist_Input.VP_Int_Total, RF_base_Completa_Hist_Input.Dias_Liq
FROM RF_Fecha_Proceso_Carteras INNER JOIN RF_base_Completa_Hist_Input ON RF_Fecha_Proceso_Carteras.Fecha = RF_base_Completa_Hist_Input.Fec_Pro
WHERE Left(RF_base_Completa_Hist_Input.Cod_Pro,20)='Inversion Financiera' And (Right(RF_base_Completa_Hist_Input.Cod_Sub_Pro,4)='Gtia' Or Right(RF_base_Completa_Hist_Input.Cod_Sub_Pro,8)='Gtia_Liq');


   [Longitud SQL

In [26]:
# =============================================================================
# GUARDAR ANÁLISIS EN ARCHIVO MARKDOWN
# =============================================================================

output_md = """# Análisis de RF_PLI_044e_Modelo_Inversiones_Tabla_Final

## Contexto del Problema

El paso 21 del modelo de inversiones (`RF_PLI_044e_Modelo_Inversiones_Tabla_Final`) es una query que depende de manera anidada de otras 11 queries, formando una estructura compleja tipo "spaghetti code". Este documento analiza cada query, identifica patrones problemáticos y propone simplificaciones.

## Resumen de Dependencias

```
RF_PLI_001b_CarteraInv_Gtia
    └── RF_PLI_001c_CarteraInv_Gtia
    
RF_PLI_008b_CarteraGobCLP_Final
RF_PLI_015b_CarteraGobCLF_Final  
RF_PLI_022b_CarteraDPF_Final
RF_PLI_029b_CarteraDPR_Final
RF_PLI_036b_CarteraLCH_Final
RF_PLI_043b_CarteraBBC_Final
    └── RF_PLI_044_Modelo_Inversiones_Final (UNION de todas las carteras)
    
RF_PLI_044c_Modelo_Inversiones_Pacto_FB
    └── RF_PLI_044d_Modelo_Inversiones_Full (UNION con pactos)
        └── RF_PLI_044e_Modelo_Inversiones_Tabla_Final (SELECT INTO final)
```

---

## Análisis Detallado de Cada Query

"""

# Agregar cada query al markdown
for query_name in queries_paso_21:
    query_row = queries_raw[queries_raw['nombre'] == query_name]
    if not query_row.empty:
        sql = query_row['sql'].values[0]
        tipo = query_row['tipo'].values[0] if 'tipo' in query_row.columns else 'N/A'
        
        output_md += f"""
### {query_name}

**Tipo:** {tipo}  
**Longitud SQL:** {len(sql)} caracteres

```sql
{sql}
```

---
"""

# Agregar sección de análisis y recomendaciones
output_md += """
## Problemas Identificados

### 1. **Repetición Excesiva de Columnas**
Las queries listan cada columna individualmente en lugar de usar `SELECT *` o definir listas de columnas reutilizables. Esto genera:
- SQL extremadamente largo (algunas queries superan los 2000 caracteres)
- Dificultad de mantenimiento
- Mayor probabilidad de errores tipográficos

### 2. **Queries Intermedias Innecesarias**
Varias queries solo filtran o renombran columnas sin agregar lógica de negocio real. Por ejemplo:
- `RF_PLI_001b_CarteraInv_Gtia` → `RF_PLI_001c_CarteraInv_Gtia` solo hace transformaciones menores
- Las queries `*_Final` para cada instrumento tienen estructura idéntica

### 3. **Patrón UNION Repetitivo**
`RF_PLI_044_Modelo_Inversiones_Final` hace UNION de 7 queries con estructura casi idéntica, repitiendo las mismas columnas 7 veces.

### 4. **Nomenclatura Inconsistente**
Los sufijos `_Final`, `_Full`, `_Tabla_Final` no tienen un significado claro y consistente.

### 5. **Falta de Parametrización**
Los filtros están hardcodeados en el SQL en lugar de ser parámetros.

---

## Por Qué Alguien Escribió Este Código Así

### Razones Históricas y Técnicas:

1. **Evolución Incremental**: El código probablemente creció orgánicamente. Cada vez que se necesitó un nuevo reporte o modificación, se añadió una nueva query en lugar de refactorizar.

2. **Limitaciones de MS Access**: Access tiene limitaciones para crear vistas parametrizadas o funciones reutilizables, lo que incentiva copiar-pegar queries.

3. **Herencia de Excel**: El patrón de "seleccionar columna por columna" es muy común en usuarios que vienen de Excel, donde cada celda/columna se referencia individualmente.

4. **Falta de Revisión de Código**: En entornos donde no hay code review, el código tiende a acumular deuda técnica.

5. **Optimización Prematura Mal Entendida**: Algunos desarrolladores creen que "ser explícito" (listar cada columna) es mejor que usar `SELECT *`, pero lo llevan al extremo.

6. **Miedo a Romper Cosas**: Con sistemas en producción, es más "seguro" crear una nueva query que modificar una existente.

---

## Estrategia de Simplificación

### Paso 1: Identificar las Columnas Comunes
Todas las queries de cartera (`*_Final`) usan prácticamente las mismas columnas. Definir una lista constante:

```python
COLUMNAS_CARTERA = [
    'Fec_Pro', 'Cod_Emp', 'Moneda', 'Cod_Pro', 'Cod_Sub_Pro',
    'Nemotecnico', 'Instrumento', 'VP_Cap_Amort', 'VP_Int_Total',
    'Dias_Vcto', 'Monto_Liquidar', 'Haircut'
]
```

### Paso 2: Crear Función Genérica para Cartera Final
```python
def generar_cartera_final(df_flujo, tipo_instrumento, columnas=COLUMNAS_CARTERA):
    \"\"\"Genera la tabla final de cartera para cualquier instrumento\"\"\"
    df = df_flujo.copy()
    # Agregar columnas faltantes con valores por defecto
    for col in columnas:
        if col not in df.columns:
            df[col] = None
    return df[columnas]
```

### Paso 3: Unificar con un Solo UNION
```python
def generar_modelo_inversiones_final(flujos_por_instrumento):
    \"\"\"Une todas las carteras en una sola tabla\"\"\"
    return pd.concat([
        generar_cartera_final(flujo, inst)
        for inst, flujo in flujos_por_instrumento.items()
    ], ignore_index=True)
```

### Paso 4: Eliminar Queries Intermedias
En lugar de 12 queries anidadas, el flujo simplificado sería:

```
Datos Base → Flujos por Instrumento → UNION Final → Output
```

---

## Herramientas y Tips para Desenredar Código Legacy

### 1. **Análisis de Dependencias**
- Crear un grafo de dependencias (como el que hiciste en el notebook)
- Identificar queries "hoja" (sin dependencias) y queries "raíz" (de las que otros dependen)

### 2. **Comparación de Esquemas**
- Extraer las columnas de cada query
- Identificar columnas comunes vs únicas
- Usar herramientas como `pandas` para comparar estructuras

### 3. **Tests de Regresión**
Antes de simplificar, guardar los outputs actuales:
```python
# Guardar resultado original
resultado_original = ejecutar_query_access('RF_PLI_044e_...')
resultado_original.to_pickle('resultado_original.pkl')

# Después de simplificar, comparar
resultado_nuevo = funcion_simplificada()
assert resultado_nuevo.equals(resultado_original)
```

### 4. **Documentación Progresiva**
- Documentar cada query mientras la analizas
- Crear diagramas de flujo de datos
- Mantener un "diccionario" de términos del dominio

### 5. **Refactoring Incremental**
- No intentar reescribir todo de una vez
- Reemplazar una query a la vez
- Validar después de cada cambio

### 6. **Herramientas Útiles**
- **SQLGlot**: Parser de SQL que permite analizar y transformar queries
- **ERAlchemy**: Genera diagramas ER desde bases de datos
- **DBeaver**: Cliente SQL con análisis de dependencias
- **Python + Pandas**: Para prototipado rápido de la lógica

---

## Conclusión

El código "spaghetti" de Access no es malicioso ni incompetente - es el resultado natural de:
- Herramientas con limitaciones
- Presión por entregar funcionalidad
- Falta de tiempo para refactorizar
- Evolución orgánica sin arquitectura clara

La buena noticia es que la lógica subyacente es simple:
1. Tomar datos de cartera
2. Calcular métricas por instrumento
3. Unir todo en una tabla final

En Python, esto se puede expresar en ~50 líneas de código limpio en lugar de 12 queries anidadas con miles de caracteres de SQL repetitivo.
"""

# Guardar archivo
output_path = notebook_path / 'RF_PLI_044e_Modelo_Inversiones_Tabla_Final_analisis.md'
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(output_md)

print(f"✓ Análisis guardado en: {output_path}")
print(f"  Tamaño: {len(output_md):,} caracteres")

✓ Análisis guardado en: C:\Users\vlandaetat\code\bfa-cl-modelos-diarios\RF_Modelo_Inversiones\dev\RF_PLI_044e_Modelo_Inversiones_Tabla_Final_analisis.md
  Tamaño: 13,718 caracteres
